# RAG Golden Benchmark · cli-agent SDK + DeepEval

Dieses Notebook führt die zuvor generierten **Goldens gegen den tatsächlichen cli-agent** aus und bewertet seine Antworten mit **DeepEval**. Es benötigt **kein separates Python-Runner-Skript**. Die optionale Knowledge-Trace des neuen cli-agent-SDK wird automatisch für Chroma/OKF erfasst und zur Bewertung verwendet.

**Trennung:** An cli-agent wird ausschließlich `golden["input"]` gesendet. Referenzantworten, Quellenzitate und Golden-Kontext bleiben außerhalb des Agenten. Jeder Golden erhält eine **eigene SDK-Sitzung**.

**Ablauf:** Konfiguration → Goldens laden → SDK-Antworten (Checkpoint) → optionale echte Retrieval-Kontexte → DeepEval-Bewertung → Übersicht. SDK-Ausführung und Bewertung lassen sich unabhängig erneut starten.

**Installation** in deiner aktiven Notebook-Kernel-Umgebung, falls nötig:

```python
%pip install "cli-agent" "deepeval" "ipykernel"
```

DeepEval benötigt einen separat konfigurierten Judge (typischerweise `OPENAI_API_KEY` oder `JUDGE_MODEL` mit passender Provider-Konfiguration). Die Modell- und Datenschutzrichtlinien des Judge sind unabhängig von denen des RAG-Agenten.


In [ ]:
from __future__ import annotations

import argparse
import asyncio
from dataclasses import asdict, is_dataclass
import hashlib
import json
import math
import os
from pathlib import Path
import re
import tempfile
import time
import uuid
from typing import Any, Callable

MAX_GOLDENS_BYTES = 32_000_000
MAX_CASE_BYTES = 2_000_000
MAX_CONTEXT_BYTES = 16_000_000
CASE_ID = re.compile(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,99}\Z")
VERSION = 2



## 1. Konfiguration

Die Pfade beziehen sich auf **WORKSPACE** (Repository-/Projektverzeichnis mit dem RAG). Für einen sicheren Probelauf sind standardmäßig nur **3 Goldens** vorgesehen. Setze `MAX_CASES = None` für alle. Keine Secrets in Notebook-Zellen speichern.

In [ ]:
WORKSPACE = Path.cwd().resolve()
GOLDENS_FILE = WORKSPACE / "rag-goldens" / "goldens.json"
OUTPUT_DIR = WORKSPACE / "rag-goldens" / "evaluation"

# CLI-agent SDK. None nutzt die reguläre Benutzer-/Systemkonfiguration.
CONFIG_FILE = None            # z. B. WORKSPACE / "config.toml"
ADMIN_CONFIG_FILE = None      # z. B. Path("/etc/cli-agent/admin_config.toml")
APPROVE_TOOLS = ()            # exakte Toolnamen, z. B. ("okf__search",)

MAX_CASES = 3                 # None = alle Goldens
SDK_TIMEOUT_SECONDS = 600.0
FRESH_SDK = False             # SDK-Antworten trotz vorhandenem Checkpoint erneut erzeugen

# DeepEval-Judge
JUDGE_MODEL = None            # None = DeepEval-Standardmodell
THRESHOLD = 0.7
FRESH_EVALUATION = False

# Knowledge-Trace wird über das SDK automatisch erhoben.
# True erfasst verifizierte Chunks/Texte und benötigt die SDK-Erweiterung aus cli-agent PR #303.
# Ausgabe-JSON kann vertrauliche Quelleninhalte enthalten.
CAPTURE_KNOWLEDGE_CONTENT = True

# Optionaler externer Trace für Sonderfälle; normalerweise None.
OBSERVED_RETRIEVAL_FILE = None


## 2. Hilfsfunktionen, isolierte SDK-Ausführung und DeepEval

Die folgenden Zellen definieren die wiederverwendbare Implementierung. Sie führen **noch keine** LLM-Aufrufe aus. Antworten werden atomar als JSON geschrieben; Kompatibilitäts-Fingerprints helfen beim Fortsetzen. Änderungen am tatsächlichen Wissensbestand oder an externen MCP-Servern erfordern dennoch `FRESH_SDK = True` oder ein neues Ausgabeverzeichnis.

In [ ]:
def digest(value: Any) -> str:
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, ensure_ascii=False,
                   separators=(",", ":"), default=str).encode("utf-8")
    ).hexdigest()


def file_digest(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def read_json(path: Path, *, max_bytes: int) -> Any:
    if path.is_symlink() or not path.is_file() or path.stat().st_size > max_bytes:
        raise ValueError(f"Missing, symlinked or oversized JSON: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, value: Any) -> None:
    """Atomically replace checkpoints; partially written JSON is never resumed."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temp: Path | None = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", dir=path.parent, prefix=".checkpoint-",
            suffix=".tmp", delete=False,
        ) as stream:
            temp = Path(stream.name)
            os.chmod(temp, 0o600)
            json.dump(value, stream, ensure_ascii=False, indent=2)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temp, path)
    finally:
        if temp is not None:
            temp.unlink(missing_ok=True)


def load_goldens(path: Path, *, limit: int | None = None) -> list[dict[str, Any]]:
    raw = read_json(path, max_bytes=MAX_GOLDENS_BYTES)
    if not isinstance(raw, list):
        raise ValueError("goldens.json must contain a JSON array")
    if limit is not None:
        if limit < 1:
            raise ValueError("limit must be positive")
        raw = raw[:limit]
    if not raw:
        raise ValueError("No goldens to run (the generator may have found none)")
    seen = set()
    result = []
    for index, golden in enumerate(raw):
        if not isinstance(golden, dict):
            raise ValueError(f"Golden {index} is not an object")
        gid, question, expected = (
            golden.get("id"), golden.get("input"), golden.get("expected_output")
        )
        if not isinstance(gid, str) or not CASE_ID.fullmatch(gid):
            raise ValueError(f"Invalid golden id at index {index}: {gid!r}")
        if gid in seen:
            raise ValueError(f"Duplicate golden id: {gid}")
        if (not isinstance(question, str) or not question.strip()
                or len(question) > 10_000
                or not isinstance(expected, str) or not expected.strip()
                or len(expected) > 100_000):
            raise ValueError(f"Golden {gid} requires input and expected_output")
        seen.add(gid)
        result.append(golden)
    return result


def get_config_paths(config: Path | None, admin: Path | None) -> tuple[Path, Path | None]:
    # For reproducibility use a concrete SDK config path, including when the
    # caller relies on cli-agent's default config.
    if config is None:
        from cli_agent.settings.config import ensure_default_config_file
        config = ensure_default_config_file()
    config = config.expanduser().resolve(strict=True)
    if not config.is_file():
        raise ValueError("SDK project config must be a file")
    if admin is not None:
        admin = admin.expanduser().resolve(strict=True)
        if not admin.is_file():
            raise ValueError("Admin policy must be a file")
    return config, admin


def run_fingerprint(*, config: Path, admin: Path | None,
                    workspace: Path, approve_tools: tuple[str, ...],
                    capture_content: bool) -> str:
    # Hash real configuration files, not their contents in benchmark results.
    # Include project-local overrides and the default admin policy when used.
    if admin is None:
        from cli_agent.settings.admin_config import default_admin_config_file
        admin = default_admin_config_file()
    project_override = workspace / ".cli-agent" / "project.toml"
    return digest({
        "sdk": "cli-agent", "runner_version": VERSION,
        "workspace": str(workspace),
        "config_sha256": file_digest(config),
        "admin_sha256": file_digest(admin) if admin.is_file() else "not-present",
        "project_overrides_sha256": (
            file_digest(project_override) if project_override.is_file()
            else "not-present"
        ),
        "approval_names": sorted(approve_tools),
        # This changes captured SDK data and the availability of retrieval
        # evaluation metrics, even when the agent's answer is unchanged.
        "capture_knowledge_content": capture_content,
        "workspace_access": "none",
        "session_isolation": "fresh-agent-per-golden",
    })


def case_path(output: Path, golden_id: str) -> Path:
    return output / "responses" / f"{golden_id}.json"


def score_path(output: Path, golden_id: str) -> Path:
    return output / "scores" / f"{golden_id}.json"


def is_current(record: Any, *, golden_hash: str, runner_hash: str) -> bool:
    return (
        isinstance(record, dict) and record.get("status") == "ok"
        and record.get("golden_sha256") == golden_hash
        and record.get("runner_sha256") == runner_hash
        and isinstance(record.get("actual_output"), str)
        and bool(record["actual_output"].strip())
    )


def serialize_usage(value: Any) -> dict[str, Any] | None:
    if value is None:
        return None
    if is_dataclass(value):
        return asdict(value)
    return {
        key: getattr(value, key, None)
        for key in ("input_tokens", "output_tokens", "total_tokens",
                    "model_calls", "reported_calls", "unreported_calls")
    }



In [ ]:
def build_sdk_agent(*, config: Path, admin: Path | None,
                    workspace: Path, approve_tools: tuple[str, ...],
                    capture_content: bool = True) -> Any:
    from cli_agent.sdk import AgentBuilder

    builder = AgentBuilder()
    # Explicitly use the SDK's admin policy resolution: unlike the CLI,
    # SDK policies are not loaded from the system when omitted entirely.
    builder.with_admin_config(path=admin) if admin else builder.with_admin_config()
    builder.with_project_config(
        path=config, workspace=workspace, workspace_access="none",
        response_format="text",
    )
    # Instrumentation only; this never alters the Knowledge search/selection.
    # Require an SDK version which supports the new opt-in API.
    builder.with_knowledge_trace(include_content=capture_content)
    if approve_tools:
        allowed = frozenset(approve_tools)

        async def approve(name: str, arguments: dict) -> bool:
            # Exact tool names only, never wildcards. Configured tools remain
            # subject to the SDK's normal admin and workspace security rules.
            return name in allowed

        builder.with_approval_callback(approve)
    return builder.build()


async def ask_one(golden: dict[str, Any], *,
                  agent_factory: Callable[[], Any], timeout: float) -> dict[str, Any]:
    started = time.monotonic()
    # Create a NEW agent for every golden, ensuring separate conversation,
    # evidence, prior tool approvals, and reference contexts.
    agent = agent_factory()
    async with agent:
        result = await asyncio.wait_for(agent.ask(golden["input"]), timeout=timeout)
    answer = getattr(result, "answer", None)
    if not isinstance(answer, str) or not answer.strip():
        raise ValueError("SDK returned an empty answer")
    calls = []
    for call in getattr(result, "tool_calls", ()):
        calls.append({
            "name": str(call.name), "status": str(call.status),
            "duration_ms": getattr(call, "duration_ms", None),
        })
    knowledge = getattr(result, "knowledge_trace", None)
    trace = asdict(knowledge) if is_dataclass(knowledge) else None
    return {
        "actual_output": answer,
        "duration_seconds": round(time.monotonic() - started, 3),
        "tool_calls": calls,
        "usage": serialize_usage(getattr(result, "usage", None)),
        "knowledge_trace": trace,
    }


async def run_cases(
    goldens: list[dict[str, Any]], *, output: Path, runner_sha: str,
    agent_factory: Callable[[], Any], timeout: float, fresh: bool,
) -> dict[str, int]:
    summary = {"completed": 0, "reused": 0, "failed": 0}
    for pos, golden in enumerate(goldens, 1):
        gid, gh = golden["id"], digest(golden)
        path = case_path(output, gid)
        if path.exists() and not fresh:
            old = read_json(path, max_bytes=MAX_CASE_BYTES)
            if is_current(old, golden_hash=gh, runner_hash=runner_sha):
                summary["reused"] += 1
                print(f"[{pos}/{len(goldens)}] reuse {gid}", flush=True)
                continue
        print(f"[{pos}/{len(goldens)}] SDK {gid}", flush=True)
        try:
            answer = await ask_one(golden, agent_factory=agent_factory,
                                   timeout=timeout)
            record = {
                "version": VERSION, "id": gid, "status": "ok",
                "execution_id": uuid.uuid4().hex,
                "golden_sha256": gh, "runner_sha256": runner_sha,
                **answer,
            }
            summary["completed"] += 1
        except Exception as exc:
            # No sensitive exception messages (which can include model URLs,
            # server error payloads or credentials) are persisted by default.
            record = {
                "version": VERSION, "id": gid, "status": "error",
                "golden_sha256": gh, "runner_sha256": runner_sha,
                "error_type": type(exc).__name__,
            }
            summary["failed"] += 1
            print(f"  failed: {type(exc).__name__}", flush=True)
        write_json(path, record)
    return summary


def validate_actual_contexts(path: Path | None) -> dict[str, dict[str, Any]]:
    """Optional contexts observed by the ACTUAL retriever, never golden quotes.

    Contract: {"id": {"runner_sha256": "<agent run>", "chunks": ["..."]}}.
    A matching run fingerprint is required to prevent stale provenance.
    """
    if path is None:
        return {}
    raw = read_json(path, max_bytes=MAX_CONTEXT_BYTES)
    if not isinstance(raw, dict):
        raise ValueError("retrieval contexts must be a JSON object keyed by golden ID")
    for gid, value in raw.items():
        if (not isinstance(gid, str) or not CASE_ID.fullmatch(gid)
                or not isinstance(value, dict)):
            raise ValueError("Invalid retrieval context entry")
        chunks = value.get("chunks")
        if (not isinstance(value.get("runner_sha256"), str)
                or not isinstance(value.get("response_sha256"), str)
                or not re.fullmatch(r"[0-9a-f]{64}", value["response_sha256"])
                or not isinstance(chunks, list) or len(chunks) > 30
                or any(not isinstance(x, str) or not x.strip()
                       or len(x) > 15000 for x in chunks)):
            raise ValueError(f"Invalid observed chunks for {gid}")
    return raw



In [ ]:

def normalize_source_path(source: Any) -> str | None:
    """Normalize only trivial path syntax; never guess by bare basename."""
    if not isinstance(source, str) or not source.strip():
        return None
    value = source.strip().replace("\\", "/")
    if value.startswith("./"):
        value = value[2:]
    return value.rstrip("/") or None


def sdk_selected_context(response: dict[str, Any]) -> list[str] | None:
    """Only verified, complete selected passages actually sent to main agent."""
    trace = response.get("knowledge_trace")
    if not isinstance(trace, dict) or trace.get("truncated") is True:
        return None
    if trace.get("content_included") is not True:
        return None
    selected = trace.get("selected")
    if not isinstance(selected, list) or not selected:
        return None
    texts = []
    for hit in selected:
        if not isinstance(hit, dict) or not isinstance(hit.get("content"), str) or not hit["content"].strip():
            return None
        texts.append(hit["content"])
    return texts


def document_recall(golden: dict[str, Any], response: dict[str, Any]) -> dict[str, Any] | None:
    """Deterministic candidate and selected source recall, exact document IDs.

    Candidate search stages also include verified OKF reads, for OKF-only mode
    where there is no initial vector ranking.
    """
    trace = response.get("knowledge_trace")
    required = golden.get("sources")
    if not isinstance(trace, dict) or trace.get("truncated") is True:
        return None
    if not isinstance(required, list) or not required:
        return None
    required_paths = {
        normalize_source_path(s.get("path"))
        for s in required if isinstance(s, dict)
    }
    if None in required_paths or not required_paths:
        return None
    searches = trace.get("searches")
    selected = trace.get("selected")
    if not isinstance(searches, list) or not isinstance(selected, list):
        return None

    initial, all_candidates = set(), set()
    for search in searches:
        if not isinstance(search, dict):
            return None
        hits = search.get("hits", [])
        if not isinstance(hits, list):
            return None
        for hit in hits:
            if not isinstance(hit, dict):
                return None
            path = normalize_source_path(hit.get("source"))
            if path is None:
                continue
            all_candidates.add(path)
            if search.get("stage") == "initial":
                initial.add(path)
    selected_paths = set()
    for hit in selected:
        if not isinstance(hit, dict):
            return None
        path = normalize_source_path(hit.get("source"))
        if path:
            selected_paths.add(path)

    n = len(required_paths)
    missing_candidates = sorted(required_paths - all_candidates)
    missing_selected = sorted(required_paths - selected_paths)
    return {
        "required_count": n,
        "initial_recall": len(required_paths & initial) / n,
        "candidate_recall": len(required_paths & all_candidates) / n,
        "selection_recall": len(required_paths & selected_paths) / n,
        "missing_candidates": missing_candidates,
        "missing_selected": missing_selected,
        "path_match_exact": True,
    }

def create_metrics(*, threshold: float, model: str | None,
                   has_retrieval: bool) -> list[Any]:
    # Import lazily so the SDK run phase does not require DeepEval.
    from deepeval.metrics import GEval, AnswerRelevancyMetric
    from deepeval.test_case import SingleTurnParams

    kw = {"threshold": threshold, "async_mode": False}
    if model is not None:
        kw["model"] = model
    metrics: list[Any] = [
        GEval(
            name="AnswerCorrectness",
            criteria=(
                "Evaluate whether ACTUAL_OUTPUT gives a factually accurate and "
                "sufficiently complete answer to INPUT relative to EXPECTED_OUTPUT. "
                "Preserve logical qualifiers, prerequisites, exceptions and links "
                "between multiple documents. Heavily penalize omitted required "
                "facts, contradictions, unsupported specificity, or refusal when "
                "the reference has a clear answer. Accept semantically equivalent "
                "wording without requiring verbatim reproduction."
            ),
            evaluation_params=[
                SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
                SingleTurnParams.EXPECTED_OUTPUT,
            ],
            **kw,
        ),
        AnswerRelevancyMetric(**kw),
    ]
    if has_retrieval:
        from deepeval.metrics import (
            FaithfulnessMetric, ContextualRecallMetric, ContextualPrecisionMetric,
        )
        metrics.extend([
            FaithfulnessMetric(**kw),
            ContextualRecallMetric(**kw),
            ContextualPrecisionMetric(**kw),
        ])
    return metrics


def evaluate_cases(
    goldens: list[dict[str, Any]], *, output: Path, runner_sha: str,
    threshold: float, judge_model: str | None, fresh: bool,
    retrieval_contexts: dict[str, dict[str, Any]] | None = None,
    metrics_factory: Callable[..., list[Any]] = create_metrics,
    testcase_factory: Callable[..., Any] | None = None,
) -> dict[str, Any]:
    if testcase_factory is None:
        from deepeval.test_case import LLMTestCase
        testcase_factory = LLMTestCase

    contexts = retrieval_contexts or {}
    summary: dict[str, Any] = {
        "evaluated": 0, "reused": 0, "skipped": 0, "errors": 0,
        "passed": 0, "failed": 0, "scores": {}, "source_recalls": {},
    }
    for pos, golden in enumerate(goldens, 1):
        gid, gh = golden["id"], digest(golden)
        response_file = case_path(output, gid)
        if not response_file.exists():
            summary["skipped"] += 1
            continue
        response = read_json(response_file, max_bytes=MAX_CASE_BYTES)
        if not is_current(response, golden_hash=gh, runner_hash=runner_sha):
            summary["skipped"] += 1
            continue

        observed = contexts.get(gid)
        chunks = sdk_selected_context(response)
        if observed is not None:
            if (observed["runner_sha256"] != runner_sha
                    or observed["response_sha256"] != digest(response)):
                raise ValueError(f"Stale retrieval contexts for {gid}")
            # Explicit external captures take precedence only for this exact
            # execution; normal operation uses the SDK Knowledge trace.
            chunks = observed["chunks"]
        source_diagnostic = document_recall(golden, response)
        # Distinguish "not observed" from "retriever returned zero chunks".
        # An observed empty context is intentionally not sent to metrics that
        # require a nonempty grounding context; mark it explicitly.
        has_context = bool(chunks)
        eval_sha = digest({
            "golden_sha256": gh,
            "response_sha256": digest(response),
            "judge_model": judge_model or "deepeval-default",
            "threshold": threshold,
            "retrieval_sha256": digest(chunks) if chunks is not None else None,
            "metric_version": VERSION,
        })
        out = score_path(output, gid)
        if out.exists() and not fresh:
            old = read_json(out, max_bytes=MAX_CASE_BYTES)
            if old.get("evaluation_sha256") == eval_sha and old.get("status") == "ok":
                summary["reused"] += 1
                _count_score(summary, old)
                continue

        print(f"[{pos}/{len(goldens)}] DeepEval {gid}", flush=True)
        try:
            test_case = testcase_factory(
                input=golden["input"],
                expected_output=golden["expected_output"],
                actual_output=response["actual_output"],
                **({"retrieval_context": chunks} if has_context else {}),
            )
            results = {}
            for metric in metrics_factory(
                threshold=threshold, model=judge_model,
                has_retrieval=has_context,
            ):
                metric.measure(test_case)
                score = getattr(metric, "score", None)
                if (isinstance(score, bool) or not isinstance(score, (int, float))
                        or not math.isfinite(score)):
                    raise ValueError(f"Invalid DeepEval score for {metric.__class__.__name__}")
                name = str(getattr(metric, "name", None) or metric.__class__.__name__)
                if name in results:
                    raise ValueError(f"Duplicate DeepEval metric: {name}")
                results[name] = {
                    "score": float(score),
                    "threshold": threshold,
                    "passed": bool(metric.is_successful()),
                    "reason": str(getattr(metric, "reason", "") or "")[:4000],
                }
            record = {
                "version": VERSION, "id": gid, "status": "ok",
                "evaluation_sha256": eval_sha,
                "overall_passed": all(m["passed"] for m in results.values()),
                "retrieval_observed": chunks is not None,
                "retrieval_context_count": len(chunks) if chunks is not None else None,
                "source_recall": source_diagnostic,
                "knowledge_calls": (response.get("knowledge_trace") or {}).get("calls", []),
                "metrics": results,
            }
            summary["evaluated"] += 1
        except Exception as exc:
            record = {
                "version": VERSION, "id": gid, "status": "error",
                "evaluation_sha256": eval_sha,
                "error_type": type(exc).__name__,
            }
            summary["errors"] += 1
            print(f"  failed: {type(exc).__name__}", flush=True)
        write_json(out, record)
        _count_score(summary, record)

    # Avoid treating failed/missing SDK cases as a successful 100% benchmark.
    summary["total"] = len(goldens)
    summary["complete"] = (
        summary["skipped"] == 0 and summary["errors"] == 0
        and summary["evaluated"] + summary["reused"] == len(goldens)
    )
    summary["scores"] = {
        name: {"mean": round(values["sum"] / values["count"], 4),
               "count": values["count"]}
        for name, values in sorted(summary["scores"].items())
    }
    summary["source_recalls"] = {
        name: {"mean": round(values["sum"] / values["count"], 4),
               "count": values["count"]}
        for name, values in sorted(summary["source_recalls"].items())
    }
    write_json(output / "summary.json", summary)
    return summary


def _count_score(summary: dict[str, Any], record: dict[str, Any]) -> None:
    if record.get("status") != "ok":
        return
    key = "passed" if record.get("overall_passed") else "failed"
    summary[key] += 1
    for name, value in record.get("metrics", {}).items():
        agg = summary["scores"].setdefault(name, {"sum": 0.0, "count": 0})
        agg["sum"] += value["score"]
        agg["count"] += 1
    recall = record.get("source_recall")
    if isinstance(recall, dict):
        for name in ("initial_recall", "candidate_recall", "selection_recall"):
            value = recall.get(name)
            if isinstance(value, (int, float)) and not isinstance(value, bool):
                agg = summary["source_recalls"].setdefault(name, {"sum": 0.0, "count": 0})
                agg["sum"] += float(value)
                agg["count"] += 1



## 3. Goldens und Fingerprint laden

Diese Zelle liest nur die lokale Golden-Datei. Der vollständige `RUNNER_SHA` dient zur Zuordnung optionaler tatsächlich beobachteter Retrieval-Ergebnisse.

In [ ]:
GOLDENS = load_goldens(GOLDENS_FILE, limit=MAX_CASES)
CONFIG_PATH, ADMIN_PATH = get_config_paths(
    Path(CONFIG_FILE) if CONFIG_FILE is not None else None,
    Path(ADMIN_CONFIG_FILE) if ADMIN_CONFIG_FILE is not None else None,
)
RUNNER_SHA = run_fingerprint(
    config=CONFIG_PATH,
    admin=ADMIN_PATH,
    workspace=WORKSPACE,
    approve_tools=tuple(APPROVE_TOOLS),
    capture_content=CAPTURE_KNOWLEDGE_CONTENT,
)
print(f"{len(GOLDENS)} Goldens geladen")
print(f"SDK config: {CONFIG_PATH}")
print(f"Agent-Fingerprint: {RUNNER_SHA}")


## 4. SDK: Fragen beantworten

**Nur diese Zelle startet cli-agent-Modellaufrufe.** Jeder Golden bekommt einen neuen `AgentBuilder` samt eigenem Verlauf. Bereits erfolgreich gespeicherte Antworten werden bei unverändertem Fingerprint wiederverwendet. Nach einem Knowledge-/Index-/Modellupdate `FRESH_SDK = True` einstellen.

In [ ]:
SDK_RUN_SUMMARY = await run_cases(
    GOLDENS,
    output=OUTPUT_DIR,
    runner_sha=RUNNER_SHA,
    agent_factory=lambda: build_sdk_agent(
        config=CONFIG_PATH,
        admin=ADMIN_PATH,
        workspace=WORKSPACE,
        approve_tools=tuple(APPROVE_TOOLS),
        capture_content=CAPTURE_KNOWLEDGE_CONTENT,
    ),
    timeout=SDK_TIMEOUT_SECONDS,
    fresh=FRESH_SDK,
)
print(json.dumps(SDK_RUN_SUMMARY, ensure_ascii=False, indent=2))


## 5. Antworten prüfen

Die Antworten liegen einzeln unter `OUTPUT_DIR/responses/<golden-id>.json`. Referenzantworten werden erst bei der **Bewertung** genutzt. Der Fingerprint einzelner Antworten ist insbesondere für die optionale Retrieval-Instrumentierung relevant.

In [ ]:
RESPONSE_OVERVIEW = []
for golden in GOLDENS:
    file = case_path(OUTPUT_DIR, golden["id"])
    if not file.is_file():
        RESPONSE_OVERVIEW.append({"id": golden["id"], "status": "missing"})
        continue
    answer = read_json(file, max_bytes=MAX_CASE_BYTES)
    RESPONSE_OVERVIEW.append({
        "id": golden["id"],
        "status": answer.get("status", "unknown"),
        "answer_preview": str(answer.get("actual_output", ""))[:250],
        "tool_calls": len(answer.get("tool_calls", [])),
        "response_sha256": digest(answer) if answer.get("status") == "ok" else None,
        "knowledge_searches": len((answer.get("knowledge_trace") or {}).get("searches", [])),
        "selected_contexts": len((answer.get("knowledge_trace") or {}).get("selected", [])),
    })
from IPython.display import display
import pandas as pd
display(pd.DataFrame(RESPONSE_OVERVIEW))


## 6. Knowledge-Trace und Retrieval-Kontext

Mit der neuen **opt-in Knowledge-Trace** des cli-agent-SDK speichert das Notebook direkt in den Antwort-Checkpoints:
- jede initiale und zusätzliche Chroma-/OKF-Suche mit geordneten Treffern;
- tatsächliche verifizierte Knowledge-Auswahl, einschließlich Chunk-/Konzepttext;
- Knowledge-Aufrufe und ihre Dauer.

Für DeepEval verwenden wir **nur die tatsächlich ausgewählten und an den Hauptagenten übergebenen Texte** als `retrieval_context`. Wenn keine vollständigen Texte vorliegen oder die Trace `truncated=True` meldet, werden Retrieval-abhängige LLM-Metriken ausgelassen. Der Golden-Kontext wird **niemals** als Ersatz verwendet.

Aus `golden.sources` werden zusätzlich deterministische Recall-Metriken für die initial gefundenen Dokumente, alle gesuchten/gelesenen Dokumente und die final ausgewählten Dokumente berechnet. Dafür müssen die Quellpfade im Index mit denen der Goldens übereinstimmen. Quellen-IDs werden nicht anhand bloßer Dateinamen geraten.

**Optional:** `OBSERVED_RETRIEVAL_FILE` kann weiterhin eine externe Erfassung enthalten; Einträge müssen mit `runner_sha256` und `response_sha256` exakt an den Antwort-Checkpoint gebunden sein. Ohne diese Sonderoption werden die Chunks direkt aus der SDK-Trace entnommen.


In [ ]:
OBSERVED_CONTEXTS = validate_actual_contexts(
    Path(OBSERVED_RETRIEVAL_FILE) if OBSERVED_RETRIEVAL_FILE is not None else None
)
print("SDK-Knowledge-Trace aktiviert; externe Kontextdatei für",
      len(OBSERVED_CONTEXTS), "Goldens vorhanden.")


## 7. DeepEval: Antworten bewerten

**Nur diese Zelle startet Judge-Modellaufrufe.** Bewertet werden standardmäßig G-Eval **Answer Correctness** und **Answer Relevancy**. Mit nachgewiesenen abgerufenen Chunks kommen die Retrieval-Metriken hinzu. Pro Golden entstehen eigene Bewertungs-Checkpoints; der aggregierte Bericht liegt in `OUTPUT_DIR/summary.json`.

In [ ]:
EVALUATION_SUMMARY = evaluate_cases(
    GOLDENS,
    output=OUTPUT_DIR,
    runner_sha=RUNNER_SHA,
    threshold=THRESHOLD,
    judge_model=JUDGE_MODEL,
    fresh=FRESH_EVALUATION,
    retrieval_contexts=OBSERVED_CONTEXTS,
)
print(json.dumps(EVALUATION_SUMMARY, ensure_ascii=False, indent=2))


## 8. Auswertung und Fehlerlokalisierung

DeepEval bewertet Antwortkorrektheit und -relevanz und — sofern vollständige verifizierte Texte vorliegen — Faithfulness sowie Kontextmetriken. Der zusätzliche **Dokument-Recall** wird ohne Judge berechnet: initial gefunden, insgesamt gefunden/gelesen und final ausgewählt. Bei fehlendem Dokument vor der Auswahl liegt der Fehler eher in der Suche; bei gefundenem, aber verworfenem Dokument eher in der Knowledge-Auswahl. Liegen die erforderlichen Dokumente im ausgewählten Kontext und die Antwort ist falsch, ist das Haupt-LLM ein Kandidat für die Fehlerursache.

**Wichtig:** Die Pfade/Quellen-IDs aus dem Index müssen mit `golden.sources[].path` übereinstimmen, sonst ergibt die deterministische Prüfung einen falschen Null-Recall. Ein Treffer beweist nicht, dass das Haupt-LLM den Inhalt tatsächlich verwendet hat.


In [ ]:
SUMMARY_TABLE = pd.DataFrame([
    {"Metrik": metric, "Mittelwert": info["mean"], "Anzahl": info["count"]}
    for metric, info in EVALUATION_SUMMARY.get("scores", {}).items()
])
display(SUMMARY_TABLE)
SOURCE_RECALL_TABLE = pd.DataFrame([
    {"Metrik": name, "Mittelwert": info["mean"], "Anzahl": info["count"]}
    for name, info in EVALUATION_SUMMARY.get("source_recalls", {}).items()
])
display(SOURCE_RECALL_TABLE)
print("Goldens:", EVALUATION_SUMMARY["total"],
      "Bewertet:", EVALUATION_SUMMARY["evaluated"] + EVALUATION_SUMMARY["reused"],
      "Bestanden:", EVALUATION_SUMMARY["passed"],
      "Nicht bestanden:", EVALUATION_SUMMARY["failed"],
      "Fehler:", EVALUATION_SUMMARY["errors"],
      "Vollständig:", EVALUATION_SUMMARY["complete"])


## Hinweise zur Aussagekraft

- **End-to-End statt Retrieval-Oracle:** Die Golden-Antwort wird nur dem Judge, nicht dem cli-agent oder dessen Retriever gezeigt. Mit der SDK-Trace können Such- und Auswahlfehler besser getrennt werden; ob das Haupt-LLM eine Quelle wirklich verwendet hat, wird durch die Trace nicht bewiesen.
- **Benchmark-Reproduzierbarkeit:** Goldens, Konfiguration und Antwort-Checkpoints werden gehasht; Änderungen an RAG-Index, externem MCP-Dienst oder Modellgewichten werden nicht automatisch erkannt. Für einen sauberen Vergleich `FRESH_SDK=True` oder ein anderes `OUTPUT_DIR` wählen.
- **Kosten:** Jede Frage startet eine neue SDK-Sitzung. Beim erneuten Judge-Vergleich können die gespeicherten Antworten wiederverwendet werden; einfach die DeepEval-Zelle erneut ausführen.
- **Vertraulichkeit:** Antworten, Judge-Begründungen und optionale Retrieval-Chunks enthalten möglicherweise sensible Inhalte und sollten nicht eingecheckt werden.
- **Prüfbarkeit:** Die Notebook-Zellen ohne Modellaufrufe sind in `tests/test_rag_benchmark.py` mit Mocks testbar; ein vollständiger Live-Durchlauf erfordert deinen konfigurierten RAG- und Judge-Zugang.
